# 04 — Evaluación y Comparativa
NCF Híbrido vs SVD Baseline — métricas Hit Rate@10 y NDCG@10

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataset import load_and_filter, binarize, build_tag_features, split_by_user
from model import NCFHybrid
from metrics import hit_rate_at_k, ndcg_at_k, evaluate_svd_baseline
from config import EMBED_DIM, TOP_K, N_CANDIDATES, RESULTS_DIR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

## Cargar datos y modelo entrenado

In [ ]:
df = binarize(load_and_filter())

user_ids      = sorted(df['userID'].unique())
artist_ids    = sorted(df['artistID'].unique())
user_to_idx   = {u: i for i, u in enumerate(user_ids)}
artist_to_idx = {a: i for i, a in enumerate(artist_ids)}

tag_matrix, _ = build_tag_features(artist_ids)
train_df, test_df = split_by_user(df)

n_users   = len(user_ids)
n_artists = len(artist_ids)

# Cargar mejor modelo
model = NCFHybrid(n_users=n_users, n_artists=n_artists, embed_dim=EMBED_DIM).to(device)
model.load_state_dict(torch.load(RESULTS_DIR / 'best_model.pt', map_location=device))
model.eval()
print('Modelo NCF cargado desde results/best_model.pt')

## Preparar pares de test

In [ ]:
# Pares (user_idx, artist_idx) para evaluación NCF
test_pairs_ncf = [
    (user_to_idx[row.userID], artist_to_idx[row.artistID])
    for _, row in test_df.iterrows()
    if row.userID in user_to_idx and row.artistID in artist_to_idx
]

# Pares (user_id, artist_id) para evaluación SVD (IDs originales)
test_pairs_svd = [
    (row.userID, row.artistID)
    for _, row in test_df.iterrows()
]

# Usar muestra para velocidad en CPU
EVAL_SAMPLE = min(500, len(test_pairs_ncf))
np.random.seed(42)
sample_idx = np.random.choice(len(test_pairs_ncf), EVAL_SAMPLE, replace=False)
test_pairs_ncf_sample = [test_pairs_ncf[i] for i in sample_idx]
test_pairs_svd_sample = [test_pairs_svd[i] for i in sample_idx]

print(f'Evaluando sobre {EVAL_SAMPLE} pares de test (de {len(test_pairs_ncf)} totales)')

## Evaluación NCF Híbrido

In [ ]:
print('Evaluando NCF Híbrido...')
ncf_hr   = hit_rate_at_k(model, test_pairs_ncf_sample, tag_matrix, k=TOP_K, device=device)
ncf_ndcg = ndcg_at_k(model,    test_pairs_ncf_sample, tag_matrix, k=TOP_K, device=device)
print(f'NCF  — Hit Rate@{TOP_K}: {ncf_hr:.4f} | NDCG@{TOP_K}: {ncf_ndcg:.4f}')

## Evaluación SVD Baseline

In [ ]:
from sklearn.preprocessing import normalize
import scipy.sparse as sp

# Reconstruir matriz usuario-artista (log1p) para SVD
rows = [user_to_idx[u] for u in train_df['userID'] if u in user_to_idx]
cols = [artist_to_idx[a] for a in train_df['artistID'] if a in artist_to_idx]
data = np.log1p([w for u, w in zip(train_df['userID'], train_df['weight']) if u in user_to_idx])
user_item_log = sp.csr_matrix((data, (rows, cols)), shape=(n_users, n_artists)).toarray()

print('Evaluando SVD Baseline...')
svd_results = evaluate_svd_baseline(
    user_item_log, test_pairs_svd_sample,
    user_to_idx, artist_to_idx, k=TOP_K
)
print(f'SVD  — Hit Rate@{TOP_K}: {svd_results["hit_rate"]:.4f} | NDCG@{TOP_K}: {svd_results["ndcg"]:.4f}')

## Tabla comparativa

In [ ]:
results = pd.DataFrame({
    'Modelo':          ['SVD Baseline', 'NCF Híbrido'],
    f'Hit Rate@{TOP_K}': [svd_results['hit_rate'], ncf_hr],
    f'NDCG@{TOP_K}':     [svd_results['ndcg'],     ncf_ndcg],
})
print(results.to_string(index=False))

# Mejora relativa
hr_gain   = (ncf_hr   - svd_results['hit_rate']) / svd_results['hit_rate'] * 100 if svd_results['hit_rate'] > 0 else 0
ndcg_gain = (ncf_ndcg - svd_results['ndcg'])     / svd_results['ndcg']     * 100 if svd_results['ndcg']     > 0 else 0
print(f'\nMejora NCF vs SVD — Hit Rate: {hr_gain:+.1f}% | NDCG: {ndcg_gain:+.1f}%')

## Gráfica comparativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
models = ['SVD Baseline', 'NCF Híbrido']
colors = ['steelblue', 'coral']

axes[0].bar(models, [svd_results['hit_rate'], ncf_hr], color=colors, alpha=0.85)
axes[0].set_title(f'Hit Rate@{TOP_K}'); axes[0].set_ylim(0, 1)
for i, v in enumerate([svd_results['hit_rate'], ncf_hr]):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

axes[1].bar(models, [svd_results['ndcg'], ncf_ndcg], color=colors, alpha=0.85)
axes[1].set_title(f'NDCG@{TOP_K}'); axes[1].set_ylim(0, 1)
for i, v in enumerate([svd_results['ndcg'], ncf_ndcg]):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.suptitle('Comparativa: SVD Baseline vs NCF Híbrido', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada en results/comparativa_modelos.png')

## Conclusiones
- El **NCF Híbrido** incorpora información semántica de tags que el SVD ignora
- El protocolo de evaluación es idéntico para ambos (100 candidatos, mismo split)
- Las limitaciones del SVD identificadas en Sesión 3 (cold-start, datos ricos sin usar) son abordadas parcialmente por el NCF
- **Trabajo futuro**: incorporar red social (user_friends) como regularización o grafo de conocimiento